# LeetCode 2021 — Brightest Position on Street bằng Python

Link đề: https://leetcode.com/problems/brightest-position-on-street/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Prefix Sum, Ordered Set, Line Sweep

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def brightestPosition(self, lights: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển mỗi đèn `[pos, r]` thành **đoạn đóng** `[pos − r, pos + r]` và hiểu "độ sáng" = số đoạn chứa một điểm.
2. Dùng **quét đường thẳng** với mảng hiệu thưa: `+1` tại đầu đoạn, `−1` tại **ngay sau** cuối đoạn (`R + 1`) vì đoạn là **đóng**.
3. Nhận ra vị trí sáng nhất nhỏ nhất luôn là **đầu trái** của một đoạn nào đó ⇒ chỉ cần xét các điểm sự kiện.
4. Cài thêm cách **sắp riêng đầu trái / đầu phải + tìm nhị phân** để đếm số đoạn chứa một điểm.
5. Tránh các bẫy: nhầm đoạn nửa mở, thứ tự xử lý sự kiện cùng tọa độ, cập nhật đáp án khi **bằng** max.

## 2. Đề bài & ví dụ

Một con phố là trục số. `lights[i] = [position_i, range_i]` là một đèn đặt ở `position_i`, chiếu sáng đoạn **đóng** `[position_i − range_i, position_i + range_i]`.

**Độ sáng** của vị trí `p` là số đèn chiếu tới `p`. Trả về vị trí **sáng nhất**; nếu có nhiều, trả về vị trí **nhỏ nhất**.

**Ví dụ 1:** `[[-3,2],[1,2],[3,3]]` → `-1`. Các đoạn: `[-5,-1]`, `[-1,3]`, `[0,6]`. Vị trí `-1` sáng 2; các vị trí `0..3` cũng sáng 2; nhỏ nhất là `-1`.

**Ví dụ 2:** `[[1,0],[0,1]]` → `1`. Đoạn `[1,1]` và `[-1,1]`; vị trí `1` sáng 2.

**Ví dụ 3:** `[[1,2]]` → `-1`. Đoạn `[-1,3]`, mọi điểm sáng 1, nhỏ nhất `-1`.

**Ràng buộc:** `1 ≤ len(lights) ≤ 10⁵`, `−10⁸ ≤ position_i ≤ 10⁸`, `0 ≤ range_i ≤ 10⁸`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from collections import defaultdict
from typing import List

examples = [([[-3, 2], [1, 2], [3, 3]], -1), ([[1, 0], [0, 1]], 1), ([[1, 2]], -1)]
for lights, exp in examples:
    print(lights, "-> đoạn", [[p - r, p + r] for p, r in lights], "-> kỳ vọng", exp)

[[-3, 2], [1, 2], [3, 3]] -> đoạn [[-5, -1], [-1, 3], [0, 6]] -> kỳ vọng -1
[[1, 0], [0, 1]] -> đoạn [[1, 1], [-1, 1]] -> kỳ vọng 1
[[1, 2]] -> đoạn [[-1, 3]] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Đoạn đóng `[L, R]` trên số nguyên

Đoạn đóng chứa cả `L` và `R`: các số nguyên `L, L+1, ..., R` ⇒ `range(L, R + 1)`. Đèn có `range = 0` vẫn chiếu sáng đúng một điểm.

In [2]:
L, R = -1, 1
print(list(range(L, R + 1)), "| đèn range 0 tại 5 chiếu:", list(range(5, 5 + 1)))

[-1, 0, 1] | đèn range 0 tại 5 chiếu: [5]


### 3.2 Mảng hiệu cho đoạn đóng: `+1` tại `L`, `−1` tại `R + 1`

Cộng dồn mảng hiệu từ trái sang phải cho giá trị tại mỗi điểm. Vì `R` vẫn được chiếu sáng, lượng `−1` phải đặt ở `R + 1`.

In [3]:
diff = defaultdict(int)
for L, R in [(-5, -1), (-1, 3), (0, 6)]:
    diff[L] += 1; diff[R + 1] -= 1
cur = 0
for x in sorted(diff):
    cur += diff[x]
    print(f"từ x={x:2d} trở đi (tới sự kiện kế tiếp): độ sáng = {cur}")

từ x=-5 trở đi (tới sự kiện kế tiếp): độ sáng = 1
từ x=-1 trở đi (tới sự kiện kế tiếp): độ sáng = 2
từ x= 0 trở đi (tới sự kiện kế tiếp): độ sáng = 2
từ x= 4 trở đi (tới sự kiện kế tiếp): độ sáng = 1
từ x= 7 trở đi (tới sự kiện kế tiếp): độ sáng = 0


### 3.3 Đếm số đoạn chứa `p` bằng hai mảng đã sắp

Số đoạn chứa `p` = (số đoạn có `L ≤ p`) − (số đoạn có `R < p`). Với `Ls`, `Rs` đã sắp: `bisect_right(Ls, p) − bisect_left(Rs, p)`.

In [4]:
segs = [(-5, -1), (-1, 3), (0, 6)]
Ls = sorted(L for L, _ in segs); Rs = sorted(R for _, R in segs)
for p in [-1, 0, 4, 7]:
    print(f"p={p}: L<=p: {bisect_right(Ls, p)}, R<p: {bisect_left(Rs, p)} -> độ sáng {bisect_right(Ls, p) - bisect_left(Rs, p)}")

p=-1: L<=p: 2, R<p: 0 -> độ sáng 2
p=0: L<=p: 3, R<p: 1 -> độ sáng 2
p=4: L<=p: 3, R<p: 2 -> độ sáng 1
p=7: L<=p: 3, R<p: 3 -> độ sáng 0


### 3.4 Chọn max, hòa thì lấy nhỏ nhất

Duyệt các vị trí theo thứ tự **tăng dần** và chỉ cập nhật khi **lớn hơn hẳn** (`>`), thì khi hòa ta giữ vị trí đến trước (nhỏ hơn). Hoặc dùng `min(..., key=lambda p: (-bright(p), p))`.

In [5]:
vals = {-1: 2, 0: 2, 1: 2, 5: 1}
best = None
for p in sorted(vals):
    if best is None or vals[p] > vals[best]:
        best = p
print(best, "| cách khác:", min(vals, key=lambda p: (-vals[p], p)))

-1 | cách khác: -1


### 3.5 Sắp xếp tuple và thứ tự sự kiện cùng tọa độ

Python sắp tuple theo **thứ tự từ điển**: so phần tử đầu, hòa thì so phần tử thứ hai... Với sự kiện `(x, delta)`, `sorted` đặt `(x, -1)` **trước** `(x, +1)`. Đây đúng là thứ tự an toàn khi ta so sánh sau từng sự kiện: tại tọa độ `x`, các đèn **hết sáng** được trừ trước, rồi mới cộng các đèn **bắt đầu sáng**, nên không sinh "độ sáng ảo". Dùng `dict` thì không cần bận tâm vì mọi thay đổi tại `x` đã được cộng gộp.

In [6]:
events = [(1, +1), (1, -1), (0, +1), (2, -1)]
print("sorted mặc định:", sorted(events))
print("ưu tiên +1 trước (nguy hiểm):", sorted(events, key=lambda e: (e[0], -e[1])))

sorted mặc định: [(0, 1), (1, -1), (1, 1), (2, -1)]
ưu tiên +1 trước (nguy hiểm): [(0, 1), (1, 1), (1, -1), (2, -1)]


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 1 trên trục số (mỗi dòng là một đèn, `#` = được chiếu):

```
x:      -5 -4 -3 -2 -1  0  1  2  3  4  5  6
đèn 0:   #  #  #  #  #
đèn 1:               #  #  #  #  #
đèn 2:                  #  #  #  #  #  #  #
độ sáng: 1  1  1  1  2  2  2  2  2  1  1  1
```

Độ sáng chỉ **tăng** tại đầu trái của một đoạn và chỉ **giảm** ngay sau đầu phải. Vậy vị trí đầu tiên đạt độ sáng tối đa phải là nơi độ sáng vừa **tăng** — tức là một đầu trái `L_i`.

⇒ Chỉ cần xét các điểm sự kiện, không cần quét mọi số nguyên (có thể tới `4·10⁸` số).

In [7]:
lights = examples[0][0]
segs = [(p - r, p + r) for p, r in lights]
for x in range(-5, 7):
    print(f"{x:3d}: " + "#" * sum(L <= x <= R for L, R in segs))

 -5: #
 -4: #
 -3: #
 -2: #
 -1: ##
  0: ##
  1: ##
  2: ##
  3: ##
  4: #
  5: #
  6: #


## 5. Cách 1 — Brute Force / baseline: quét mọi số nguyên

**Ý tưởng:** với mỗi `x` từ `min(L)` tới `max(R)`, đếm số đoạn chứa `x`. Lấy `x` nhỏ nhất có độ sáng lớn nhất.

In [8]:
def brightest_brute(lights):
    segs = [(p - r, p + r) for p, r in lights]
    lo = min(L for L, _ in segs); hi = max(R for _, R in segs)
    best_x, best_v = None, -1
    for x in range(lo, hi + 1):
        v = sum(1 for L, R in segs if L <= x <= R)
        if v > best_v:
            best_x, best_v = x, v
    return best_x

for lights, exp in examples:
    got = brightest_brute(lights); print(got, "kỳ vọng", exp); assert got == exp

-1 kỳ vọng -1
1 kỳ vọng 1
-1 kỳ vọng -1


**Độ phức tạp:** `O(W · m)` với `W = max(R) − min(L) + 1` (tới `4·10⁸ + 1`) và `m` = số đèn. Không thể chạy với ràng buộc thật; chỉ dùng để kiểm tra với tọa độ nhỏ.

**Nút thắt:** (1) quét quá nhiều điểm trong khi giá trị chỉ đổi tại `≤ 2m` điểm; (2) mỗi điểm lại đếm lại từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** vị trí sáng nhất nhỏ nhất `p*` là đầu trái của một đoạn nào đó. Chứng minh: nếu `p*` không phải đầu trái nào, thì mọi đoạn chứa `p*` có `L < p*`, nên cũng chứa `p* − 1` ⇒ độ sáng tại `p* − 1` ≥ tại `p*` — mâu thuẫn với tính nhỏ nhất.

**Insight thuật toán (quét):** độ sáng là hàm bậc thang chỉ đổi tại các điểm `L_i` (tăng) và `R_i + 1` (giảm). Duyệt các điểm sự kiện tăng dần, cộng dồn `diff` **tại cùng một tọa độ** rồi mới so sánh với max.

**Quy tắc "nếu… thì…":**
- **Nếu** sau khi áp dụng mọi thay đổi tại `x`, độ sáng `cur > best` **thì** `ans = x`, `best = cur`.
- **Nếu** `cur == best` **thì** giữ `ans` cũ (nhỏ hơn).

**Invariant:** sau khi xử lý tọa độ sự kiện `x`, `cur` = độ sáng trên `[x, x_tiếp_theo − 1]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ xét các đầu trái, đếm trực tiếp (`O(m²)`)

Áp dụng bổ đề nhưng đếm độ sáng tại mỗi `L_i` bằng một vòng lặp.

In [9]:
def brightest_candidates(lights):
    segs = [(p - r, p + r) for p, r in lights]
    best_x, best_v = None, -1
    for x in sorted({L for L, _ in segs}):
        v = sum(1 for L, R in segs if L <= x <= R)
        if v > best_v:
            best_x, best_v = x, v
    return best_x

for lights, exp in examples:
    assert brightest_candidates(lights) == exp
print([brightest_candidates(l) for l, _ in examples])

[-1, 1, -1]


**Độ phức tạp Cách 2:** `O(m²)` — `10¹⁰` với `m = 10⁵`, vẫn quá chậm, nhưng đã độc lập với độ rộng tọa độ.

### 7.2 Cách 3 — Quét với `dict` hiệu (bản nộp chính)

**Giải thích từng đoạn:**
1. `diff[p − r] += 1`, `diff[p + r + 1] −= 1`.
2. Duyệt `x` trong `sorted(diff)`, `cur += diff[x]` (đã gộp mọi sự kiện cùng tọa độ nhờ `dict`).
3. Nếu `cur > best`: cập nhật.

In [10]:
def brightest_sweep(lights):
    diff = defaultdict(int)
    for p, r in lights:
        diff[p - r] += 1
        diff[p + r + 1] -= 1
    cur = best = 0
    ans = None
    for x in sorted(diff):
        cur += diff[x]
        if cur > best:
            best, ans = cur, x
    return ans

for lights, exp in examples:
    assert brightest_sweep(lights) == exp
print([brightest_sweep(l) for l, _ in examples])

[-1, 1, -1]


In [11]:
class SolutionSweep:
    def brightestPosition(self, lights: List[List[int]]) -> int:
        diff = defaultdict(int)
        for p, r in lights:
            diff[p - r] += 1
            diff[p + r + 1] -= 1
        cur = best = 0
        ans = 0
        for x in sorted(diff):
            cur += diff[x]
            if cur > best:
                best, ans = cur, x
        return ans

print([SolutionSweep().brightestPosition(l) for l, _ in examples])

[-1, 1, -1]


**Độ phức tạp Cách 3:** `O(m log m)` (sắp `≤ 2m` khóa), Space `O(m)`.

### 7.3 Cách 4 — Sắp riêng `Ls`, `Rs` + tìm nhị phân tại mỗi đầu trái

Độ sáng tại `x` = `#(L ≤ x) − #(R < x)`. Duyệt các `L` tăng dần (theo bổ đề) và tính bằng `bisect`.

In [12]:
def brightest_bisect(lights):
    Ls = sorted(p - r for p, r in lights)
    Rs = sorted(p + r for p, r in lights)
    best_x, best_v = None, -1
    for x in Ls:                                   # ứng viên: các đầu trái (tăng dần)
        v = bisect_right(Ls, x) - bisect_left(Rs, x)
        if v > best_v:
            best_x, best_v = x, v
    return best_x

for lights, exp in examples:
    assert brightest_bisect(lights) == exp
print([brightest_bisect(l) for l, _ in examples])

[-1, 1, -1]


**Độ phức tạp Cách 4:** sắp `O(m log m)`, mỗi ứng viên `O(log m)` ⇒ `O(m log m)`. Space `O(m)`.

### 7.4 Cách 5 — Hai con trỏ trên `Ls` và `Rs`

Thay `bisect` bằng hai con trỏ: khi `x` tăng dần (duyệt `Ls`), con trỏ `j` trên `Rs` chỉ tiến lên để đếm số `R < x`. `i + 1` là số `L ≤ x` khi `x = Ls[i]` (lấy chỉ số cuối cùng của các `L` bằng nhau).

In [13]:
def brightest_two_pointers(lights):
    Ls = sorted(p - r for p, r in lights); Rs = sorted(p + r for p, r in lights)
    m = len(Ls); j = 0; best_x, best_v = None, -1
    for i in range(m):
        if i + 1 < m and Ls[i + 1] == Ls[i]:
            continue                               # đợi tới L cuối cùng trong nhóm bằng nhau
        x = Ls[i]
        while j < m and Rs[j] < x:
            j += 1
        v = (i + 1) - j
        if v > best_v:
            best_x, best_v = x, v
    return best_x

for lights, exp in examples:
    assert brightest_two_pointers(lights) == exp
print([brightest_two_pointers(l) for l, _ in examples])

[-1, 1, -1]


**Độ phức tạp Cách 5:** `O(m log m)` do sắp xếp; phần quét `O(m)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1. Đèn 0 cho `+1` tại `−5` và `−1` tại `0`; đèn 1 cho `+1` tại `−1` và `−1` tại `4`; đèn 2 cho `+1` tại `0` và `−1` tại `7`. Tại `x = 0` hai thay đổi triệt tiêu: `diff[0] = −1 + 1 = 0`.

| x | diff[x] | cur | best | ans |
|---|---|---|---|---|
| −5 | +1 | 1 | 1 | −5 |
| −1 | +1 | 2 | 2 | **−1** |
| 0 | 0 | 2 | 2 | −1 (hòa, giữ) |
| 4 | −1 | 1 | 2 | −1 |
| 7 | −1 | 0 | 2 | −1 |

In [14]:
def trace_sweep(lights):
    diff = defaultdict(int)
    for p, r in lights:
        diff[p - r] += 1; diff[p + r + 1] -= 1
    print("diff =", dict(sorted(diff.items())))
    cur = best = 0; ans = None
    for x in sorted(diff):
        cur += diff[x]
        note = ""
        if cur > best:
            best, ans = cur, x; note = "<- max mới"
        elif cur == best:
            note = "(hòa, giữ vị trí nhỏ hơn)"
        print(f"x={x:3d}: diff={diff[x]:+d}, cur={cur}, best={best}, ans={ans} {note}")
    return ans

print("Đáp án:", trace_sweep(examples[0][0]))

diff = {-5: 1, -1: 1, 0: 0, 4: -1, 7: -1}
x= -5: diff=+1, cur=1, best=1, ans=-5 <- max mới
x= -1: diff=+1, cur=2, best=2, ans=-1 <- max mới
x=  0: diff=+0, cur=2, best=2, ans=-1 (hòa, giữ vị trí nhỏ hơn)
x=  4: diff=-1, cur=1, best=2, ans=-1 
x=  7: diff=-1, cur=0, best=2, ans=-1 
Đáp án: -1


In [15]:
print("Đáp án:", trace_sweep(examples[1][0]))

diff = {-1: 1, 1: 1, 2: -2}
x= -1: diff=+1, cur=1, best=1, ans=-1 <- max mới
x=  1: diff=+1, cur=2, best=2, ans=1 <- max mới
x=  2: diff=-2, cur=0, best=2, ans=1 
Đáp án: 1


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def brightestPosition(self, lights: List[List[int]]) -> int:
        diff = defaultdict(int)
        for p, r in lights:
            diff[p - r] += 1            # bắt đầu được chiếu sáng
            diff[p + r + 1] -= 1        # đoạn đóng: hết sáng từ R + 1
        cur = best = 0
        ans = 0
        for x in sorted(diff):          # mọi thay đổi tại x đã được gộp
            cur += diff[x]
            if cur > best:              # '>' để hòa thì giữ vị trí nhỏ hơn
                best, ans = cur, x
        return ans

for lights, exp in examples:
    assert Solution().brightestPosition(lights) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Hàm bậc thang.** Gọi `B(x)` là độ sáng tại `x`. Với mảng hiệu `diff`, `B(x) = Σ_{t ≤ x} diff[t]` (đèn `i` đóng góp `+1` cho mọi `t ≥ L_i` và `−1` cho mọi `t ≥ R_i + 1`, tổng là `1` đúng khi `L_i ≤ x ≤ R_i`).
2. **Bất biến quét.** Sau khi xử lý khóa `x_j`, `cur = B(x_j)`, và `B` không đổi trên `[x_j, x_{j+1} − 1]`.
3. **Đáp án.** Giá trị lớn nhất của `B` đạt được trên một đoạn `[x_j, x_{j+1} − 1]` nào đó, và điểm nhỏ nhất của đoạn đó là chính khóa `x_j`. Thuật toán ghi nhận khóa **đầu tiên** đạt max (do dùng `>`), đó là vị trí sáng nhất nhỏ nhất. (Khóa dạng `R + 1` không thể là đáp án vì tại đó `B` giảm — điều kiện `>` không bao giờ được kích hoạt lần đầu ở đó, phù hợp với bổ đề ở mục 6.)

In [17]:
# Kiểm chứng bổ đề: vị trí sáng nhất nhỏ nhất luôn là một đầu trái L_i
rng = random.Random(10)
for _ in range(500):
    lights = [[rng.randint(-10, 10), rng.randint(0, 5)] for _ in range(rng.randint(1, 8))]
    assert brightest_brute(lights) in {p - r for p, r in lights}
print("Bổ đề đúng trên 500 bộ đèn ngẫu nhiên")

Bổ đề đúng trên 500 bộ đèn ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `m` = số đèn, `W` = độ rộng vùng tọa độ (tới `4·10⁸`).

| Cách | Time | Space |
|---|---|---|
| 1. Quét mọi số nguyên | `O(W · m)` | `O(m)` |
| 2. Ứng viên đầu trái, đếm thẳng | `O(m²)` | `O(m)` |
| 3. Quét `dict` hiệu | `O(m log m)` | `O(m)` |
| 4. Sắp `Ls`, `Rs` + bisect | `O(m log m)` | `O(m)` |
| 5. Hai con trỏ | `O(m log m)` | `O(m)` |

In [18]:
rng = random.Random(2021)
big = [[rng.randint(-10**8, 10**8), rng.randint(0, 10**8)] for _ in range(10**5)]
for name, fn in [("sweep", brightest_sweep), ("bisect", brightest_bisect), ("two_ptr", brightest_two_pointers),
                 ("Solution", lambda l: Solution().brightestPosition(l))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:8s}: {r} ({time.perf_counter() - t:.3f}s)")

sweep   : -971537 (0.107s)


bisect  : -971537 (0.065s)
two_ptr : -971537 (0.058s)


Solution: -971537 (0.105s)


## 12. Bộ test

Ví dụ đề; biên (một đèn range 0; đèn trùng nhau; hai đoạn chỉ chạm nhau tại một điểm; đoạn rời nhau cùng độ sáng; tọa độ âm); đối chiếu mọi cách với brute force trên dữ liệu ngẫu nhiên nhỏ (seed cố định).

In [19]:
fns = {
    "brute": brightest_brute, "candidates": brightest_candidates, "sweep": brightest_sweep,
    "bisect": brightest_bisect, "two_ptr": brightest_two_pointers,
    "Solution": lambda l: Solution().brightestPosition(l), "SolutionSweep": lambda l: SolutionSweep().brightestPosition(l),
}
edge = [
    ([[5, 0]], 5),
    ([[0, 1], [0, 1], [0, 1]], -1),
    ([[0, 1], [2, 1]], 1),                   # [-1,1] và [1,3] chạm nhau tại 1
    ([[0, 0], [10, 0]], 0),                  # rời nhau, hòa -> nhỏ nhất
    ([[-100, 3], [-50, 60]], -103),
    ([[0, 0], [1, 0]], 0),                   # [0,0] và [1,1]: không chồng
]
for lights, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(lights) == exp, (name, lights, fn(lights))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [20]:
rng = random.Random(42)
for _ in range(800):
    lights = [[rng.randint(-15, 15), rng.randint(0, rng.choice([0, 2, 6]))] for _ in range(rng.randint(1, 10))]
    want = brightest_brute(lights)
    for name, fn in fns.items():
        assert fn(lights) == want, (name, lights)
print("800 test ngẫu nhiên: mọi cách khớp brute force")

800 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi đoạn là nửa mở: `−1` tại `R` thay vì `R + 1`

Ví dụ 2: đèn `[1, 0]` chiếu đúng điểm `1`; nếu đặt `−1` tại `1` thì đèn này "biến mất".

In [21]:
def wrong_half_open(lights):
    diff = defaultdict(int)
    for p, r in lights:
        diff[p - r] += 1; diff[p + r] -= 1          # SAI: phải là p + r + 1
    cur = best = 0; ans = None
    for x in sorted(diff):
        cur += diff[x]
        if cur > best: best, ans = cur, x
    return ans

lights = examples[1][0]
print("sai:", wrong_half_open(lights), "| đúng:", examples[1][1])

sai: -1 | đúng: 1


In [22]:
print("sửa:", brightest_sweep(lights))

sửa: 1


### Lỗi 2 — Cập nhật khi `cur >= best` (hòa thì lấy vị trí lớn hơn)

Ví dụ 1 sẽ trả `0` thay cho `-1`.

In [23]:
def wrong_ge(lights):
    diff = defaultdict(int)
    for p, r in lights:
        diff[p - r] += 1; diff[p + r + 1] -= 1
    cur = best = 0; ans = None
    for x in sorted(diff):
        cur += diff[x]
        if cur >= best: best, ans = cur, x          # SAI: >=
    return ans

lights = examples[0][0]
print("sai:", wrong_ge(lights), "| đúng:", examples[0][1])

sai: 0 | đúng: -1


In [24]:
print("sửa (>):", brightest_sweep(lights))

sửa (>): -1


### Lỗi 3 — Danh sách sự kiện, xử lý `+1` trước `−1` tại cùng tọa độ rồi so sánh ngay

Nếu dùng danh sách sự kiện và so sánh sau **từng** sự kiện, thứ tự trong cùng tọa độ quan trọng. Đèn `[0,0]` (hết sáng từ 1) và đèn `[1,0]` (bắt đầu tại 1): xử lý `+1` trước sẽ tạo độ sáng ảo `2` tại `1`.

In [25]:
def wrong_event_order(lights):
    events = []
    for p, r in lights:
        events.append((p - r, +1)); events.append((p + r + 1, -1))
    events.sort(key=lambda e: (e[0], -e[1]))        # SAI: +1 trước -1, và so sánh sau từng sự kiện
    cur = best = 0; ans = None
    for x, d in events:
        cur += d
        if cur > best: best, ans = cur, x
    return ans

lights = [[0, 0], [1, 0]]
print("sai:", wrong_event_order(lights), "| đúng:", brightest_brute(lights))

sai: 1 | đúng: 0


In [26]:
def fixed_event_order(lights):
    events = []
    for p, r in lights:
        events.append((p - r, +1)); events.append((p + r + 1, -1))
    events.sort()                                   # (x, -1) đứng trước (x, +1)
    cur = best = 0; ans = None
    for x, d in events:
        cur += d
        if cur > best: best, ans = cur, x
    return ans

print("sửa (−1 trước +1):", fixed_event_order(lights), "| hoặc dùng dict:", brightest_sweep(lights))

sửa (−1 trước +1): 0 | hoặc dùng dict: 0


### Lỗi 4 — Chỉ xét vị trí các **đèn** (`position_i`) làm ứng viên

Vị trí sáng nhất không nhất thiết là chỗ đặt đèn. Ví dụ 1: đáp án `-1` không phải vị trí đèn nào (`-3, 1, 3`).

In [27]:
def wrong_lamp_positions(lights):
    segs = [(p - r, p + r) for p, r in lights]
    return min((p for p, _ in lights), key=lambda x: (-sum(L <= x <= R for L, R in segs), x))   # SAI

lights = examples[0][0]
print("sai:", wrong_lamp_positions(lights), "| đúng:", examples[0][1])

sai: 1 | đúng: -1


In [28]:
print("sửa (ứng viên là các đầu trái):", brightest_candidates(lights))

sửa (ứng viên là các đầu trái): -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Điểm bị phủ bởi nhiều đoạn nhất"** ⇒ quét đường thẳng: `+1` đầu, `−1` sau cuối.
- **Đoạn đóng trên số nguyên** ⇒ `−1` tại `R + 1`; **đoạn nửa mở** `[L, R)` ⇒ `−1` tại `R`.
- Tọa độ lớn ⇒ `dict` + `sorted(keys)`, hoặc sắp riêng `Ls`, `Rs`.
- **"Nếu hòa lấy nhỏ nhất"** + duyệt tăng dần ⇒ cập nhật bằng `>`.
- Max của hàm bậc thang luôn đạt tại một điểm **tăng** ⇒ chỉ cần ứng viên là đầu trái.

## 15. Biến thể của bài

1. **LeetCode 1094 — Car Pooling / 1109 — Corporate Flight Bookings:** mảng hiệu trên tọa độ nhỏ.
2. **LeetCode 253 — Meeting Rooms II (Premium):** số phòng tối thiểu = số khoảng chồng nhau tối đa (khoảng nửa mở).
3. **LeetCode 2251 — Number of Flowers in Full Bloom:** nhiều truy vấn "điểm `p` bị phủ bởi bao nhiêu đoạn" ⇒ Cách 4 (`bisect` trên `Ls`, `Rs`).
4. **Đèn có độ sáng khác nhau** (trọng số `w_i`): `+w` / `−w` thay cho `±1`.
5. **Số vị trí nguyên đạt độ sáng tối đa** (bài tập 5).

Minh họa biến thể 3 (nhiều truy vấn, tự dựng, kiểm bằng brute force):

In [29]:
def brightness_queries(lights, queries):
    Ls = sorted(p - r for p, r in lights); Rs = sorted(p + r for p, r in lights)
    return [bisect_right(Ls, q) - bisect_left(Rs, q) for q in queries]

rng = random.Random(2251)
for _ in range(200):
    lights = [[rng.randint(-10, 10), rng.randint(0, 4)] for _ in range(rng.randint(1, 8))]
    qs = [rng.randint(-16, 16) for _ in range(10)]
    assert brightness_queries(lights, qs) == [sum(p - r <= q <= p + r for p, r in lights) for q in qs]
print("Truy vấn độ sáng bằng bisect khớp brute force; ví dụ 1 tại -2..2 ->",
      brightness_queries(examples[0][0], range(-2, 3)))

Truy vấn độ sáng bằng bisect khớp brute force; ví dụ 1 tại -2..2 -> [1, 2, 2, 2, 2]


Minh họa biến thể 4 — **đèn có trọng số** (độ sáng mỗi đèn là `w_i`, tự dựng): chỉ cần thay `±1` bằng `±w`. Lưu ý với trọng số dương, bổ đề "đáp án là một đầu trái" vẫn đúng.

In [30]:
def brightest_weighted(lamps):
    diff = defaultdict(int)
    for p, r, w in lamps:
        diff[p - r] += w; diff[p + r + 1] -= w
    cur = best = 0; ans = None
    for x in sorted(diff):
        cur += diff[x]
        if cur > best: best, ans = cur, x
    return ans

def brightest_weighted_brute(lamps):
    lo = min(p - r for p, r, _ in lamps); hi = max(p + r for p, r, _ in lamps)
    return max(range(lo, hi + 1), key=lambda x: (sum(w for p, r, w in lamps if p - r <= x <= p + r), -x))

rng = random.Random(44)
for _ in range(300):
    lamps = [[rng.randint(-8, 8), rng.randint(0, 4), rng.randint(1, 5)] for _ in range(rng.randint(1, 7))]
    assert brightest_weighted(lamps) == brightest_weighted_brute(lamps)
print("Biến thể có trọng số khớp brute force; ví dụ:", brightest_weighted([[0, 2, 1], [5, 1, 3], [1, 0, 1]]))

Biến thể có trọng số khớp brute force; ví dụ: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét mọi số nguyên | brute force | `O(W·m)` | `O(m)` | kiểm tra |
| 2. Đầu trái + đếm thẳng | bổ đề ứng viên | `O(m²)` | `O(m)` | bước trung gian |
| 3. `dict` hiệu | quét, gộp cùng tọa độ | `O(m log m)` | `O(m)` | **nộp bài** |
| 4. `Ls`, `Rs` + bisect | `#L≤x − #R<x` | `O(m log m)` | `O(m)` | nhiều truy vấn điểm |
| 5. Hai con trỏ | như 4, không bisect | `O(m log m)` | `O(m)` | khi thích hai con trỏ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `[[0,0],[10,0],[5,5]]`.
2. **Tính tay:** `[[2,1],[4,1],[3,0]]` — lập bảng `diff`, `cur`, `best`, `ans`.
3. **Điền TODO:** `brightness_at(lights, p)` trả về độ sáng tại `p` trong `O(m)`.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `count_brightest(lights)` — số **vị trí nguyên** có độ sáng bằng độ sáng tối đa (trong `O(m log m)`). Kiểm bằng brute force.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def brightness_at(lights, p):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_brightest(lights):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_brightest(lights):
    # TODO: quét dict hiệu; khi cur == best, cộng độ dài đoạn [x, x_next - 1]
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Đoạn: `[0,0]`, `[10,10]`, `[0,10]`. (2) Điểm `0` và `10` sáng 2. (3) Lấy nhỏ nhất `0`.

**Bài 2.** (1) Đoạn `[1,3]`, `[3,5]`, `[3,3]`. (2) `diff = {1:+1, 3:+2, 4:−2, 6:−1}`. (3) `cur` tại 3 là 3 ⇒ đáp án `3`.

**Bài 3.** (1) Duyệt đèn. (2) Đếm `p − r ≤ p_q ≤ p + r`. (3) `sum(...)`.

**Bài 4.** (1) `diff[L] += 1`, `diff[R + 1] −= 1`. (2) Duyệt khóa tăng dần. (3) Cập nhật khi `>`.

**Bài 5.** (1) Lần quét thứ nhất tìm `best`. (2) Lần quét thứ hai: với mỗi cặp khóa liên tiếp `(x, y)`, nếu độ sáng trên `[x, y−1]` bằng `best` thì cộng `y − x`. (3) Khóa cuối luôn có độ sáng 0, không cần xét.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().brightestPosition([[0, 0], [10, 0], [5, 5]]) == 0
assert Solution().brightestPosition([[2, 1], [4, 1], [3, 0]]) == 3
print("Bài 1: 0 | Bài 2: 3")
trace_sweep([[2, 1], [4, 1], [3, 0]])

Bài 1: 0 | Bài 2: 3
diff = {1: 1, 3: 2, 4: -2, 6: -1}
x=  1: diff=+1, cur=1, best=1, ans=1 <- max mới
x=  3: diff=+2, cur=3, best=3, ans=3 <- max mới
x=  4: diff=-2, cur=1, best=3, ans=3 
x=  6: diff=-1, cur=0, best=3, ans=3 


3

In [35]:
# Bài 3, 4
def brightness_at_ans(lights, p):
    return sum(1 for q, r in lights if q - r <= p <= q + r)

my_brightest_ans = brightest_sweep
rng = random.Random(3)
for _ in range(300):
    lights = [[rng.randint(-8, 8), rng.randint(0, 4)] for _ in range(rng.randint(1, 7))]
    ans = my_brightest_ans(lights)
    lo = min(p - r for p, r in lights); hi = max(p + r for p, r in lights)
    vals = [brightness_at_ans(lights, x) for x in range(lo, hi + 1)]
    assert brightness_at_ans(lights, ans) == max(vals) and ans == lo + vals.index(max(vals))
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [36]:
# Bài 5
def count_brightest_ans(lights):
    diff = defaultdict(int)
    for p, r in lights:
        diff[p - r] += 1; diff[p + r + 1] -= 1
    keys = sorted(diff)
    cur, levels = 0, []
    for x in keys:
        cur += diff[x]; levels.append(cur)
    best = max(levels)
    return sum(keys[i + 1] - keys[i] for i in range(len(keys) - 1) if levels[i] == best)

def count_brightest_brute(lights):
    lo = min(p - r for p, r in lights); hi = max(p + r for p, r in lights)
    vals = [brightness_at_ans(lights, x) for x in range(lo, hi + 1)]
    return vals.count(max(vals))

rng = random.Random(5)
for _ in range(500):
    lights = [[rng.randint(-10, 10), rng.randint(0, 5)] for _ in range(rng.randint(1, 8))]
    assert count_brightest_ans(lights) == count_brightest_brute(lights)
print("Bài 5: OK; ví dụ 1 ->", count_brightest_ans(examples[0][0]), "vị trí (−1, 0, 1, 2, 3)")

Bài 5: OK; ví dụ 1 -> 5 vị trí (−1, 0, 1, 2, 3)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "điểm bị phủ nhiều nhất, hòa lấy nhỏ nhất":**

```python
diff = defaultdict(int)
for L, R in segments:          # đoạn đóng
    diff[L] += 1; diff[R + 1] -= 1
cur = best = 0; ans = None
for x in sorted(diff):
    cur += diff[x]
    if cur > best: best, ans = cur, x
return ans
```

In [37]:
def template(segments):
    diff = defaultdict(int)
    for L, R in segments:
        diff[L] += 1; diff[R + 1] -= 1
    cur = best = 0; ans = None
    for x in sorted(diff):
        cur += diff[x]
        if cur > best: best, ans = cur, x
    return ans
print([template([(p - r, p + r) for p, r in l]) for l, _ in examples])

[-1, 1, -1]


**Checklist:**

- [ ] Đoạn là `[p − r, p + r]` **đóng** ⇒ `−1` tại `p + r + 1`.
- [ ] Gộp mọi thay đổi tại cùng tọa độ trước khi so sánh (dùng `dict`, hoặc sắp `−1` trước `+1`).
- [ ] Cập nhật bằng `>` để hòa thì lấy vị trí nhỏ nhất.
- [ ] Ứng viên là đầu trái của đoạn, không phải vị trí đèn.
- [ ] Đèn `range = 0` vẫn chiếu một điểm.
- [ ] Đã thử: tọa độ âm, đèn trùng, đoạn chạm nhau tại một điểm.